# AutoEIT — Specific Test I: Audio-to-Text Transcription Pipeline

**GSoC 2026 Application | AutoEIT Project**  
**Task:** Generate transcriptions of 4 Spanish EIT audio files (30 sentences each)  
**Approach:** Whisper large-v3 + DP sequence alignment (zero API cost, fully reproducible)

---

## How to run this notebook

**Step 1.** In Colab: `Runtime > Change runtime type > T4 GPU` (free, ~15GB VRAM)  
**Step 2.** Upload these 5 files when Cell 3 asks:  
- `AutoEIT_Sample_Audio_Transcriptions.xlsx`  
- `038010_EIT-2A.mp3`  
- `038011_EIT-1A.mp3`  
- `038012_EIT-2A.mp3`  
- `038015_EIT-1A.mp3`  

**Step 3.** Run all cells top to bottom (`Runtime > Run all`).  
**Step 4.** Two output files will auto-download at the end:  
- `AutoEIT_Transcriptions_FILLED.xlsx` — filled-in template with color-coded confidence  
- `AutoEIT_Full_Transcription_Report.csv` — detailed quality metrics per sentence  

> **Runtime estimate:** ~15-20 min total on T4 GPU (Whisper large-v3 is thorough)

In [5]:
# =============================================================================
# CELL 1 — Install dependencies
# =============================================================================
# faster-whisper: CTranslate2-backed reimplementation of Whisper.
#   Same model weights as openai-whisper but ~2-4x faster on GPU.
#   Zero API cost — model downloads once and runs entirely locally.
# noisereduce: spectral noise gating to clean up room noise before ASR.
# jiwer: standard WER/CER metrics used in ASR research.
# python-Levenshtein: C-extension edit distance, much faster than difflib.
# openpyxl: reading and writing the .xlsx output template.
# pydub + soundfile: audio I/O and format conversion.

!pip install -q faster-whisper noisereduce jiwer python-Levenshtein openpyxl pydub soundfile librosa

# ffmpeg is Whisper's audio decoder — usually pre-installed on Colab.
# The 2>/dev/null suppresses the apt progress spam.
!apt-get install -y -qq ffmpeg 2>/dev/null

print("All dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 104.2 MB/s eta 0:00:00
All dependencies installed.


In [6]:
# =============================================================================
# CELL 2 — Imports, constants, and EIT metadata
# =============================================================================
import os, re, shutil, json, unicodedata, warnings
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import noisereduce as nr
from pydub import AudioSegment
from faster_whisper import WhisperModel
from jiwer import wer, cer
import Levenshtein
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from google.colab import files
from IPython.display import display, HTML

warnings.filterwarnings('ignore')

# ── EIT stimulus list ─────────────────────────────────────────────────────────
# These are the 30 target sentences from the Excel (same for all 4 participants
# because the Info sheet confirms: "All files are EIT vA (same stimuli)").
# The syllable count (from the Excel) is stored alongside the text because
# we use it later as a sanity-check weight in the alignment cost function.
TARGET_STIMULI = [
    ("Quiero cortarme el pelo", 7),
    ("El libro está en la mesa", 7),
    ("El carro lo tiene Pedro", 8),
    ("El se ducha cada mañana", 9),
    ("¿Qué dice usted que va a hacer hoy?", 9),
    ("Dudo que sepa manejar muy bien", 10),
    ("Las calles de esta ciudad son muy anchas", 11),
    ("Puede que llueva mañana todo el día", 12),
    ("Las casas son muy bonitas pero caras", 12),
    ("Me gustan las películas que acaban bien", 12),
    ("El chico con el que yo salgo es español", 13),
    ("Después de cenar me fui a dormir tranquilo", 13),
    ("Quiero una casa en la que vivan mis animales", 14),
    ("A nosotros nos fascinan las fiestas grandiosas", 14),
    ("Ella sólo bebe cerveza y no come nada", 15),
    ("Me gustaría que el precio de las casas bajara", 15),
    ("Cruza a la derecha y después sigue todo recto", 15),
    ("Ella ha terminado de pintar su apartamento", 15),
    ("Me gustaría que empezara a hacer más calor pronto", 15),
    ("El niño al que se le murió el gato está triste", 16),
    ("Una amiga mía cuida a los niños de mi vecino", 16),
    ("El gato que era negro fue perseguido por el perro", 16),
    ("Antes de poder salir él tiene que limpiar su cuarto", 16),
    ("La cantidad de personas que fuman ha disminuido", 17),
    ("Después de llegar a casa del trabajo tomé la cena", 17),
    ("El ladrón al que atrapó la policía era famoso", 17),
    ("Le pedí a un amigo que me ayudara con la tarea", 17),
    ("El examen no fue tan difícil como me habían dicho", 17),
    ("¿Serías tan amable de darme el libro que está en la mesa?", 17),
    ("Hay mucha gente que no toma nada para el desayuno", 17),
]

# ── Participant registry ──────────────────────────────────────────────────────
# offset_sec: seconds to skip at the START of the audio before the EIT begins.
# This is 720s (12:00) for 038012 per the explicit note in the Excel sheet.
PARTICIPANTS = {
    "38010": {"file": "038010_EIT-2A.mp3", "version": "2A", "sheet": "38010-2A", "offset_sec": 0},
    "38011": {"file": "038011_EIT-1A.mp3", "version": "1A", "sheet": "38011-1A", "offset_sec": 0},
    "38012": {"file": "038012_EIT-2A.mp3", "version": "2A", "sheet": "38012-2A", "offset_sec": 720},
    "38015": {"file": "038015_EIT-1A.mp3", "version": "1A", "sheet": "38015-1A", "offset_sec": 0},
}

# ── Path constants ─────────────────────────────────────────────────────────────
EXCEL_TEMPLATE  = "AutoEIT_Sample_Audio_Transcriptions.xlsx"
OUTPUT_EXCEL    = "AutoEIT_Transcriptions_FILLED.xlsx"
OUTPUT_CSV      = "AutoEIT_Full_Transcription_Report.csv"
PROCESSED_DIR   = "processed_audio"
os.makedirs(PROCESSED_DIR, exist_ok=True)

N_STIMULI = len(TARGET_STIMULI)  # always 30
print(f"EIT config loaded: {N_STIMULI} target stimuli, {len(PARTICIPANTS)} participants.")
print("Stimuli syllable range:", TARGET_STIMULI[0][1], "→", TARGET_STIMULI[-1][1])

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


EIT config loaded: 30 target stimuli, 4 participants.
Stimuli syllable range: 7 → 17


In [7]:
# =============================================================================
# CELL 3 — Upload files to the Colab session
# =============================================================================
# Upload all 5 files at once using the dialog that appears below.
# Colab lets you select multiple files in one go (Ctrl+click on Mac/Linux,
# Cmd+click on Windows in the browser file picker).

print("Please upload ALL of the following files:")
print("  1. AutoEIT_Sample_Audio_Transcriptions.xlsx")
print("  2. 038010_EIT-2A.mp3")
print("  3. 038011_EIT-1A.mp3")
print("  4. 038012_EIT-2A.mp3")
print("  5. 038015_EIT-1A.mp3")
print()

uploaded = files.upload()
print(f"\nSuccessfully uploaded:")
for fname, data in uploaded.items():
    print(f"  {fname}  ({len(data)/1024:.0f} KB)")

# Quick sanity check — make sure all required files landed
required = [EXCEL_TEMPLATE] + [info["file"] for info in PARTICIPANTS.values()]
missing  = [f for f in required if not os.path.exists(f)]
if missing:
    print(f"\n⚠️  Missing files (re-upload and re-run this cell): {missing}")
else:
    print("\n✓ All required files present. Ready to proceed.")

Please upload ALL of the following files:
  1. AutoEIT_Sample_Audio_Transcriptions.xlsx
  2. 038010_EIT-2A.mp3
  3. 038011_EIT-1A.mp3
  4. 038012_EIT-2A.mp3
  5. 038015_EIT-1A.mp3



Saving AutoEIT_Sample_Audio_Transcriptions.xlsx to AutoEIT_Sample_Audio_Transcriptions.xlsx
Saving 038010_EIT-2A.mp3 to 038010_EIT-2A.mp3
Saving 038011_EIT-1A.mp3 to 038011_EIT-1A.mp3
Saving 038012_EIT-2A.mp3 to 038012_EIT-2A.mp3
Saving 038015_EIT-1A.mp3 to 038015_EIT-1A.mp3

Successfully uploaded:
  AutoEIT_Sample_Audio_Transcriptions.xlsx  (22 KB)
  038010_EIT-2A.mp3  (9093 KB)
  038011_EIT-1A.mp3  (9360 KB)
  038012_EIT-2A.mp3  (18599 KB)
  038015_EIT-1A.mp3  (8796 KB)

✓ All required files present. Ready to proceed.


In [8]:
# =============================================================================
# CELL 4 — Audio preprocessing
# =============================================================================
# Why we preprocess instead of feeding raw MP3s directly to Whisper:
#
#  1. Offset trimming (038012 only): the Excel note says "relevant audio begins
#     at 12:00". The first 12 minutes contain setup/preamble. Passing that
#     to Whisper would waste compute and risk hallucinated segments.
#     IMPORTANT: if the audio is shorter than the stated offset we clamp to
#     a safe fallback rather than crashing or silently trimming everything.
#
#  2. 16 kHz mono WAV: Whisper's internal resampler uses a fast but low-quality
#     algorithm. Doing the conversion ourselves with librosa's high-quality
#     resampler means better waveform fidelity — this matters for L2 speech
#     where subtle phonetic distinctions determine whether Whisper mishears.
#
#  3. Light spectral denoising: we estimate the noise floor from a 0.5-second
#     quiet window and apply spectral gating with prop_decrease=0.4.
#     We deliberately keep this MILD — aggressive denoising can blur consonants
#     like /s/, /t/, /p/ which are exactly the sounds L2 learners most often
#     get wrong. We only want to suppress stationary background hum.

def preprocess_audio(input_path, output_path, offset_sec=0, target_sr=16000):
    """
    Load an MP3, optionally trim a leading preamble, apply mild spectral
    denoising, and save as 16 kHz mono WAV ready for Whisper.
    Returns the output path for chaining.
    """
    print(f"  Loading {os.path.basename(input_path)} ...")
    audio = AudioSegment.from_mp3(input_path)
    original_dur_sec = len(audio) / 1000.0
    print(f"  Total audio duration: {original_dur_sec/60:.1f} min ({original_dur_sec:.0f}s)")

    if offset_sec > 0:
        trim_mm = offset_sec // 60
        trim_ss = offset_sec % 60

        # Safety guard: if stated offset is >= total duration, the trim value
        # is wrong. Clamp to leave at least the last 25% of the file, and warn
        # loudly so the user knows to check the offset manually.
        if offset_sec >= original_dur_sec:
            safe_offset = int(original_dur_sec * 0.75)
            print(f"  ⚠️  WARNING: offset_sec={offset_sec}s but audio is only "
                  f"{original_dur_sec:.0f}s long!")
            print(f"  ⚠️  Clamping offset to {safe_offset}s (75% of file). "
                  f"Verify the correct offset manually.")
            offset_sec = safe_offset

        audio = audio[offset_sec * 1000:]   # pydub works in milliseconds
        remaining_dur = len(audio) / 1000.0
        print(f"  Trimmed {trim_mm}:{trim_ss:02d} of preamble. "
              f"Remaining: {remaining_dur:.0f}s ({remaining_dur/60:.1f} min)")

        if remaining_dur < 60:
            raise ValueError(
                f"After trimming, only {remaining_dur:.0f}s of audio remains — "
                f"that's too short for a 30-sentence EIT. "
                f"Check offset_sec in PARTICIPANTS for {os.path.basename(input_path)}."
            )

    # Convert to 16 kHz mono before hitting librosa so the resampling is
    # done by pydub (which uses ffmpeg) rather than librosa's resampler twice.
    audio = audio.set_channels(1).set_frame_rate(target_sr)

    # Dump to a temp WAV so librosa can read it as float32
    tmp_wav = output_path.replace(".wav", "_tmp.wav")
    audio.export(tmp_wav, format="wav")
    y, sr = librosa.load(tmp_wav, sr=target_sr, mono=True)
    os.remove(tmp_wav)

    # Use a half-second quiet window (right at the start, after the trim)
    # as the noise profile. If that window is shorter than expected (very
    # short remaining audio), skip denoising rather than crash.
    noise_frames = min(int(0.5 * sr), len(y) // 8)
    if noise_frames > 256:
        y_clean = nr.reduce_noise(
            y=y,
            sr=sr,
            y_noise=y[:noise_frames],
            prop_decrease=0.40,       # mild — preserve speech texture
            stationary=True,          # only attack constant-frequency noise
        )
    else:
        y_clean = y  # audio is too short to safely estimate noise

    sf.write(output_path, y_clean, sr, subtype="PCM_16")
    duration_min = len(y_clean) / sr / 60
    print(f"  ✓ {os.path.basename(output_path)}  [{duration_min:.1f} min of usable audio]")
    return output_path


print("Preprocessing audio files...\n")
processed_paths = {}
for pid, meta in PARTICIPANTS.items():
    in_path  = meta["file"]
    out_path = os.path.join(PROCESSED_DIR, f"{pid}_clean.wav")
    if os.path.exists(in_path):
        processed_paths[pid] = preprocess_audio(in_path, out_path, meta["offset_sec"])
    else:
        print(f"  SKIP {in_path} — not found (did you upload it in Cell 3?)")
    print()

print(f"Preprocessing done. {len(processed_paths)}/{len(PARTICIPANTS)} files ready.")


Preprocessing audio files...

  Loading 038010_EIT-2A.mp3 ...
  Total audio duration: 9.2 min (550s)
  ✓ 38010_clean.wav  [9.2 min of usable audio]

  Loading 038011_EIT-1A.mp3 ...
  Total audio duration: 9.1 min (547s)
  ✓ 38011_clean.wav  [9.1 min of usable audio]

  Loading 038012_EIT-2A.mp3 ...
  Total audio duration: 18.8 min (1126s)
  Trimmed 12:00 of preamble. Remaining: 406s (6.8 min)
  ✓ 38012_clean.wav  [6.8 min of usable audio]

  Loading 038015_EIT-1A.mp3 ...
  Total audio duration: 8.8 min (529s)
  ✓ 38015_clean.wav  [8.8 min of usable audio]

Preprocessing done. 4/4 files ready.


In [9]:
# =============================================================================
# CELL 5 — Load Whisper large-v3
# =============================================================================
# Model choice rationale:
#
# large-v3 is the only Whisper variant that consistently handles:
#   - L2 Spanish with non-native phonology (accented vowels, stops, fricatives)
#   - Disfluencies (false starts, hesitations, syllable repetitions)
#   - Low-to-intermediate proficiency speech with unpredictable syntax
#
# The smaller models (base, small, medium) frequently hallucinate or produce
# "corrected" versions of what the learner said — exactly the wrong behavior
# for EIT transcription where learner errors are the signal, not the noise.
#
# compute_type=float16: safe on Colab T4's 15 GB VRAM, roughly 2x faster than
# float32 with negligible accuracy difference for transcription tasks.
# Falls back to int8 on CPU (slower but runs without a GPU).

import torch
HAS_GPU      = torch.cuda.is_available()
DEVICE       = "cuda" if HAS_GPU else "cpu"
COMPUTE_TYPE = "float16" if HAS_GPU else "int8"

if HAS_GPU:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU detected: {gpu_name}  ({vram_gb:.1f} GB VRAM)")
else:
    print("No GPU detected — running on CPU (much slower).")
    print("  → In Colab: Runtime > Change runtime type > T4 GPU")

print(f"\nLoading Whisper large-v3 ({COMPUTE_TYPE} on {DEVICE.upper()}) ...")
print("First run will download ~3 GB of model weights. Cached on subsequent runs.")

whisper_model = WhisperModel(
    "large-v3",
    device=DEVICE,
    compute_type=COMPUTE_TYPE,
    cpu_threads=4,          # only relevant when falling back to CPU
    num_workers=2,
)

print("✓ Whisper large-v3 loaded and ready.")

GPU detected: Tesla T4  (15.6 GB VRAM)

Loading Whisper large-v3 (float16 on CUDA) ...
First run will download ~3 GB of model weights. Cached on subsequent runs.


✓ Whisper large-v3 loaded and ready.


In [10]:
# =============================================================================
# CELL 6 — Whisper transcription (EIT-specific settings)
# =============================================================================
# language="es":
#   Force Spanish decoding. Without this, Whisper's language detection can
#   occasionally misclassify heavily accented L2 speech as a different language.
#
# condition_on_previous_text=False:  ← THE MOST IMPORTANT SETTING FOR EIT
#   Whisper's default feeds the previous segment as a language-model prefix for
#   the next. For EIT this is harmful: it biases decoding toward what fluently
#   follows the last transcription rather than what the learner actually said.
#
# temperature=0.0:
#   Deterministic decoding. Higher temperatures cause hallucination in the long
#   inter-stimulus silences characteristic of EIT recordings.
#
# vad_filter=True:
#   Silero VAD marks long silences as non-speech. Critical for EIT because
#   3–8s inter-stimulus gaps would otherwise generate hallucinated text.
#
# no_speech_threshold=0.65:
#   Segments where Whisper's own no-speech probability exceeds this are dropped.
#   Slightly above default (0.6) to accept more borderline L2 speech segments.
#
# word_timestamps=True:
#   Word-level timing for better merge decisions and the timing report.

EIT_INITIAL_PROMPT = (
    "Transcripción de una tarea de imitación provocada en español. "
    "El participante repite frases cortas con posibles errores gramaticales y disfluencias. "
    "No corregir errores del hablante."
)

# Build a normalised version of the prompt for echo detection (see below).
_PROMPT_NORMALIZED = EIT_INITIAL_PROMPT.lower().replace(" ", "")

def _is_prompt_echo(text):
    """
    Return True if Whisper hallucinated the initial_prompt instead of
    transcribing real speech. This happens when a segment is near-silent:
    Whisper's decoder falls back to the most probable token sequence given
    the prompt, which is... the prompt itself.

    We detect it by checking if a large fraction of the segment text (after
    lowercasing and stripping spaces) appears inside the normalised prompt.
    A 40% overlap threshold catches full echoes and partial echoes while
    being loose enough not to flag short Spanish sentences that happen to
    share common words with the prompt.
    """
    if not text:
        return False
    norm = text.lower().replace(" ", "")
    # Sliding-window: check if any 40%-length substring of norm is in the prompt
    window = max(1, int(len(norm) * 0.40))
    for start in range(len(norm) - window + 1):
        if norm[start:start + window] in _PROMPT_NORMALIZED:
            return True
    return False


def transcribe_participant(audio_path, participant_id):
    """
    Run Whisper on one participant's preprocessed audio.
    Returns a list of raw segment dicts with prompt-echo segments removed.
    """
    print(f"\n  Transcribing participant {participant_id} ...")

    segments_iter, audio_info = whisper_model.transcribe(
        audio_path,
        language="es",
        beam_size=5,
        best_of=5,
        temperature=0.0,
        initial_prompt=EIT_INITIAL_PROMPT,
        condition_on_previous_text=False,
        vad_filter=True,
        vad_parameters=dict(
            min_silence_duration_ms=500,   # shorter than default to detect EIT boundaries
            speech_pad_ms=400,             # more padding — soft L2 voices need it
            threshold=0.25,                # more sensitive for quiet/hesitant learners
        ),
        no_speech_threshold=0.65,
        word_timestamps=True,
        compression_ratio_threshold=2.4,
        log_prob_threshold=-1.0,           # keep weak segments — human will review them
    )

    raw_segments = []
    n_echo = 0
    for seg in segments_iter:
        text = seg.text.strip()

        # ── Prompt-echo filter ────────────────────────────────────────────────
        # Whisper sometimes echoes the initial_prompt when it hears silence.
        # These are not real transcriptions — discard them before alignment.
        if _is_prompt_echo(text):
            n_echo += 1
            print(f"    [echo filter] Prompt echo discarded at "
                  f"{seg.start:.1f}s–{seg.end:.1f}s: {text[:60]!r}")
            continue

        raw_segments.append({
            "text":              text,
            "start":             round(seg.start, 3),
            "end":               round(seg.end, 3),
            "no_speech_prob":    round(seg.no_speech_prob, 4),
            "avg_logprob":       round(seg.avg_logprob, 4),
            "compression_ratio": round(seg.compression_ratio, 4),
            "words": [
                (w.word.strip(), round(w.start, 3), round(w.end, 3), round(w.probability, 3))
                for w in (seg.words or [])
            ],
        })

    detected_lang_prob = audio_info.language_probability
    print(f"    Language detection confidence: {detected_lang_prob:.2%}")
    print(f"    Raw Whisper segments: {len(raw_segments) + n_echo} total, "
          f"{n_echo} echo(es) removed, {len(raw_segments)} kept")

    # ── Raw segment dump — helps diagnose alignment problems ─────────────────
    print(f"\n    Raw segments passed to alignment:")
    for i, s in enumerate(raw_segments):
        print(f"      [{i:>2}] {s['start']:6.1f}s–{s['end']:6.1f}s  "
              f"nsp={s['no_speech_prob']:.2f}  lp={s['avg_logprob']:.2f}  "
              f"{s['text'][:55]!r}")

    return raw_segments


print("Starting transcription pipeline...\n" + "─" * 50)
all_raw_segments = {}
for pid, wav_path in processed_paths.items():
    all_raw_segments[pid] = transcribe_participant(wav_path, pid)

print("\n" + "─" * 50)
print("Transcription complete for all participants.")


Starting transcription pipeline...
──────────────────────────────────────────────────

  Transcribing participant 38010 ...
    Language detection confidence: 100.00%
    Raw Whisper segments: 19 total, 0 echo(es) removed, 19 kept

    Raw segments passed to alignment:
      [ 0]   73.7s– 121.7s  nsp=0.02  lp=-0.10  "We drove to the park. I'll call her tomorrow night. You"
      [ 1]  164.6s– 189.6s  nsp=0.60  lp=-0.17  'Quiero cortarme el pelo. El libro está en la mesa. El c'
      [ 2]  196.6s– 218.9s  nsp=0.60  lp=-0.17  '¿Qué dice ustedes que va a hacer hoy? Dudo que sepa man'
      [ 3]  227.1s– 229.9s  nsp=0.26  lp=-0.23  'Puede que lleve mañana todo el día.'
      [ 4]  238.3s– 242.9s  nsp=0.26  lp=-0.23  'Las casas son muy bonitas, pero caras.'
      [ 5]  250.0s– 252.7s  nsp=0.26  lp=-0.23  'Me gustan las películas que caben bien.'
      [ 6]  262.4s– 265.8s  nsp=0.26  lp=-0.23  'El chico que hago es español.'
      [ 7]  274.5s– 278.0s  nsp=0.26  lp=-0.23  'Después de cenar, 

In [11]:
# =============================================================================
# CELL 7 — EIT sequence alignment
# =============================================================================
# The alignment problem: Whisper returns K segments; we need exactly 30,
# one per EIT stimulus. K can be > 30 (Whisper over-split), < 30 (Whisper
# under-split or skipped silences), or = 30 (lucky).
#
# Algorithm: order-preserving Dynamic Programming alignment.
# Each Whisper segment (or merged pair) is assigned to exactly one stimulus
# slot; stimulus slots can also be left [inaudible] at a fixed cost.
#
# Key parameter changes from earlier versions:
#
# MAX_WORDS_PER_RESPONSE = 12
#   Any Whisper segment longer than this is split at sentence boundaries
#   BEFORE the DP sees it. This is the primary fix for multi-sentence bleeding:
#   Whisper sometimes outputs an entire paragraph as one segment when the
#   learner produces several responses with very short inter-sentence pauses.
#
# MAX_MERGE_SEGS = 1 (with conditional bump to 2 for very short pairs)
#   One Whisper segment → one stimulus slot by default. The only exception is
#   when two adjacent segments are each very short (< 5 words each) and close
#   together (< MERGE_GAP_SEC) — likely a single disfluent response split by
#   the VAD mid-word. Even then we only merge if their combined length still
#   doesn't exceed MAX_WORDS_PER_RESPONSE.
#
# INAUDIBLE_COST = 0.72
#   Slightly reduced from 0.75 to make the DP more willing to mark noisy
#   slots as [inaudible] rather than force-assigning a bad match.
#
# LENGTH_PENALTY_WEIGHT = 0.35
#   Applied to edit_ratio_with_length_penalty when the candidate text is
#   more than 1.5× longer than the stimulus. Without this, a 3-sentence
#   blob scores low edit-distance on the first sentence and gets assigned.

MAX_MERGE_SEGS          = 1     # see note above — bumped to 2 conditionally
INAUDIBLE_COST          = 0.72
MIN_SEG_DUR             = 0.35  # seconds
MIN_SEG_CHARS           = 3     # characters
MERGE_GAP_SEC           = 0.8   # seconds
MAX_WORDS_PER_RESPONSE  = 12    # hard cap before forced split
LENGTH_PENALTY_WEIGHT   = 0.35  # penalty when candidate is much longer than stimulus

_ENGLISH_KEYWORDS = re.compile(
    r'\b(drove|park|computer|volleyball|butcher|tomorrow|meat|brother|call|night|shop)\b',
    re.IGNORECASE
)
_SPANISH_FUNCTION = re.compile(
    r'\b(que|el|la|en|de|un|una|se|no|para|los|las|del|al|con|por|su|esto|esta)\b',
    re.IGNORECASE
)
# Sentence-boundary split pattern (period/question/exclamation not inside a word)
_SENT_BOUNDARY = re.compile(r'(?<=[.?!])\s+')


def normalize(text):
    """Lowercase + strip accent marks for accent-insensitive comparison."""
    nfkd = unicodedata.normalize("NFKD", text)
    return "".join(c for c in nfkd if not unicodedata.combining(c)).lower().strip()


def edit_ratio_with_length_penalty(candidate, stimulus):
    """
    Normalized edit distance with an extra penalty when the candidate is
    significantly longer than the stimulus.

    This prevents a multi-sentence Whisper blob from appearing "close" to a
    short stimulus just because the first few words happen to match.
    Without this penalty the DP happily assigns "Quiero cortarme el pelo.
    El libro está en la mesa. El carro..." to S01 because the edit distance
    to "Quiero cortarme el pelo" is low in absolute terms.
    """
    c_norm = normalize(candidate)
    s_norm = normalize(stimulus)
    if not c_norm and not s_norm: return 0.0
    if not c_norm or not s_norm:  return 1.0

    base_dist = Levenshtein.distance(c_norm, s_norm) / max(len(c_norm), len(s_norm))

    # Length ratio penalty: if candidate is more than 1.5× the stimulus length,
    # add a penalty proportional to the excess.
    len_ratio = len(c_norm) / max(len(s_norm), 1)
    if len_ratio > 1.5:
        length_penalty = LENGTH_PENALTY_WEIGHT * min(len_ratio - 1.5, 1.5)
        return min(1.0, base_dist + length_penalty)

    return base_dist


def _split_long_segment(seg):
    """
    If a segment contains more than MAX_WORDS_PER_RESPONSE words, attempt to
    split it at sentence boundaries (. ? !) into sub-segments.
    Each sub-segment inherits the parent's quality metrics; timestamps are
    distributed proportionally by word count since word-level timestamps may
    not always be reliable for a long hallucinated blob.
    Returns a list of one or more segment dicts.
    """
    words = seg["text"].split()
    if len(words) <= MAX_WORDS_PER_RESPONSE:
        return [seg]

    # Split on sentence boundaries
    sentences = _SENT_BOUNDARY.split(seg["text"].strip())
    sentences = [s.strip() for s in sentences if s.strip()]

    if len(sentences) <= 1:
        # No sentence boundary found — truncate at MAX_WORDS_PER_RESPONSE words
        # and discard the rest (the extra is almost certainly hallucination)
        truncated = " ".join(words[:MAX_WORDS_PER_RESPONSE])
        print(f"    [split] Truncated long segment ({len(words)} words → "
              f"{MAX_WORDS_PER_RESPONSE}): {truncated[:50]!r}")
        return [{**seg, "text": truncated}]

    # Distribute timestamps proportionally
    total_chars = max(sum(len(s) for s in sentences), 1)
    total_dur   = (seg["end"] or seg["start"] or 0) - (seg["start"] or 0)
    result = []
    t = seg["start"] or 0.0
    for sent in sentences:
        frac = len(sent) / total_chars
        dur  = total_dur * frac
        result.append({
            **seg,
            "text":  sent,
            "start": round(t, 3),
            "end":   round(t + dur, 3),
            "words": [],   # word-level timestamps not meaningful after split
        })
        t += dur
        print(f"    [split] Sub-segment: {sent[:55]!r}")

    return result


def merge_whisper_segments(raw_segs):
    """
    Step 1: Filter noise, English warmup, and over-long segments.
    Step 2: Conditionally merge short adjacent disfluency pairs.
    """
    # ── First pass: filter noise, English warmup, split long segments ─────────
    cleaned = []
    n_noise, n_english, n_split = 0, 0, 0
    for seg in raw_segs:
        duration = seg["end"] - seg["start"]

        is_noise = (
            duration < MIN_SEG_DUR
            or len(seg["text"]) < MIN_SEG_CHARS
            or seg["no_speech_prob"] > 0.85
            or (seg["avg_logprob"] < -2.0 and seg["compression_ratio"] > 3.0)
        )
        if is_noise:
            n_noise += 1
            continue

        is_english = (
            _ENGLISH_KEYWORDS.search(seg["text"]) is not None
            and _SPANISH_FUNCTION.search(seg["text"]) is None
        )
        if is_english:
            n_english += 1
            print(f"    [filter] English warmup discarded: {seg['text'][:60]!r}")
            continue

        # Split segments that are too long before they reach the DP
        sub_segs = _split_long_segment(seg)
        if len(sub_segs) > 1:
            n_split += 1
        cleaned.extend(sub_segs)

    print(f"    Filter summary: {n_noise} noise, {n_english} English warmup, "
          f"{n_split} over-long split → {len(cleaned)} segments kept")

    if not cleaned:
        return []

    # ── Second pass: merge ONLY short adjacent disfluency pairs ───────────────
    # We merge two adjacent segments only when BOTH are very short (≤ 5 words)
    # AND they are temporally close (gap < MERGE_GAP_SEC).
    # This handles "El libro... está en la mesa" split mid-phrase, without
    # accidentally merging two separate EIT responses.
    merged = [dict(cleaned[0])]
    for seg in cleaned[1:]:
        prev = merged[-1]
        gap  = seg["start"] - prev["end"]
        prev_words = len(prev["text"].split())
        curr_words = len(seg["text"].split())
        combined_words = prev_words + curr_words

        should_merge = (
            gap < MERGE_GAP_SEC
            and prev_words <= 5            # both halves are short — likely one response split
            and curr_words <= 5
            and combined_words <= MAX_WORDS_PER_RESPONSE
        )

        if should_merge:
            prev["text"]              = (prev["text"] + " " + seg["text"]).strip()
            prev["end"]               = seg["end"]
            prev["words"]            += seg.get("words", [])
            prev["no_speech_prob"]    = max(prev["no_speech_prob"], seg["no_speech_prob"])
            prev["avg_logprob"]       = min(prev["avg_logprob"], seg["avg_logprob"])
            prev["compression_ratio"] = max(prev["compression_ratio"], seg["compression_ratio"])
        else:
            merged.append(dict(seg))

    return merged


def dp_align(segments, stimuli, inaudible_cost=INAUDIBLE_COST):
    """
    Order-preserving DP alignment of K segments to M stimuli.
    MAX_MERGE_SEGS is effectively 1 here because over-long merging is now
    handled upstream in merge_whisper_segments. The DP only needs to decide
    between assigning one segment or skipping (inaudible).
    """
    K = len(segments)
    M = len(stimuli)
    INF = float("inf")

    dp   = [[INF] * (M + 1) for _ in range(K + 1)]
    path = [[None] * (M + 1) for _ in range(K + 1)]
    dp[0][0] = 0.0

    stim_texts = [s[0] for s in stimuli]

    for j in range(1, M + 1):
        for i in range(K + 1):
            if dp[i][j - 1] == INF:
                continue

            # Option A: skip — mark stimulus j as [inaudible]
            cost_skip = dp[i][j - 1] + inaudible_cost
            if cost_skip < dp[i][j]:
                dp[i][j]   = cost_skip
                path[i][j] = (i, j - 1, 0)

            # Option B: assign up to MAX_MERGE_SEGS segments to stimulus j
            for k in range(1, min(MAX_MERGE_SEGS + 1, K - i + 1)):
                window_text = " ".join(seg["text"] for seg in segments[i : i + k])
                cost = dp[i][j - 1] + edit_ratio_with_length_penalty(
                    window_text, stim_texts[j - 1]
                )
                ni = i + k
                if cost < dp[ni][j]:
                    dp[ni][j]   = cost
                    path[ni][j] = (i, j - 1, k)

    best_i, best_cost = K, dp[K][M]
    if best_cost == INF:
        for i in range(K - 1, -1, -1):
            if dp[i][M] < INF:
                best_i, best_cost = i, dp[i][M]
                break

    assignments = {}
    i, j = best_i, M
    while j > 0:
        if path[i][j] is None:
            break
        prev_i, prev_j, k = path[i][j]
        stim_idx = j - 1

        if k == 0:
            assignments[stim_idx] = {
                "text": "[inaudible]", "start": None, "end": None,
                "n_segs_merged": 0, "avg_logprob": None,
                "no_speech_prob": 1.0, "dp_cost": inaudible_cost,
            }
        else:
            window = segments[prev_i : i]
            assignments[stim_idx] = {
                "text":           " ".join(s["text"] for s in window),
                "start":          window[0]["start"],
                "end":            window[-1]["end"],
                "n_segs_merged":  k,
                "avg_logprob":    float(np.mean([s["avg_logprob"] for s in window])),
                "no_speech_prob": float(max(s["no_speech_prob"] for s in window)),
                "dp_cost":        round(dp[i][j] - dp[prev_i][prev_j], 4),
            }
        i, j = prev_i, prev_j

    result = []
    for idx in range(M):
        if idx in assignments:
            result.append(assignments[idx])
        else:
            result.append({
                "text": "[inaudible]", "start": None, "end": None,
                "n_segs_merged": 0, "avg_logprob": None,
                "no_speech_prob": 1.0, "dp_cost": inaudible_cost,
            })
    return result, best_cost


def run_alignment(pid, raw_segs):
    """Full alignment pipeline for one participant."""
    print(f"\n  Aligning participant {pid} ...")
    merged = merge_whisper_segments(raw_segs)
    print(f"    After merge/filter: {len(merged)} segments → {N_STIMULI} stimulus slots")

    if not merged:
        print(f"    WARNING: no usable segments after filtering.")
        return [{"text": "[inaudible]", "start": None, "end": None,
                 "n_segs_merged": 0, "avg_logprob": None,
                 "no_speech_prob": 1.0, "dp_cost": INAUDIBLE_COST}
                for _ in range(N_STIMULI)]

    aligned, total_cost = dp_align(merged, TARGET_STIMULI)
    n_inaudible = sum(1 for r in aligned if r["text"] == "[inaudible]")
    avg_cost    = total_cost / N_STIMULI
    print(f"    DP cost: total={total_cost:.3f}  avg/sentence={avg_cost:.3f}")
    print(f"    Inaudible slots: {n_inaudible}/{N_STIMULI}")

    # Alignment table — spot-check every assignment before writing to Excel
    print(f"\n    {'S':>3}  {'Time':^13}  {'Stimulus':<32}  → Transcription")
    print(f"    {'─'*95}")
    for idx, assignment in enumerate(aligned):
        stim_text  = TARGET_STIMULI[idx][0][:30]
        trans_text = assignment["text"][:42]
        time_str   = (f"{assignment['start']:.0f}s–{assignment['end']:.0f}s"
                      if assignment["start"] is not None else "[no audio]")
        flag = "⚠" if len(assignment["text"].split()) > MAX_WORDS_PER_RESPONSE else " "
        print(f"    {idx+1:>3}  {time_str:^13}  {stim_text:<32}  {flag} {trans_text}")

    return aligned


print("Running EIT sequence alignment...\n" + "─" * 50)
all_aligned = {}
for pid, raw_segs in all_raw_segments.items():
    all_aligned[pid] = run_alignment(pid, raw_segs)

print("\n" + "─" * 50)
print("Alignment complete.")


Running EIT sequence alignment...
──────────────────────────────────────────────────

  Aligning participant 38010 ...
    [filter] English warmup discarded: "We drove to the park. I'll call her tomorrow night. You can "
    [split] Sub-segment: 'Quiero cortarme el pelo.'
    [split] Sub-segment: 'El libro está en la mesa.'
    [split] Sub-segment: 'El carro no tiene pelo.'
    [split] Sub-segment: 'Él se duche cada mañana.'
    [split] Sub-segment: '¿Qué dice ustedes que va a hacer hoy?'
    [split] Sub-segment: 'Dudo que sepa manejar muy bien.'
    [split] Sub-segment: 'Las calles de esta ciudad son muy anchas.'
    [split] Truncated long segment (14 words → 12): 'El gato que era negro, antes de poder salir, tiene'
    [split] Sub-segment: 'La cantidad de personas que fuman después de llegar a c'
    [split] Sub-segment: 'El ladrón policía era famoso.'
    [split] Sub-segment: 'Le pedí a un amigo que ayudara a la tarea.'
    Filter summary: 0 noise, 1 English warmup, 3 over-long spli

In [12]:
def edit_ratio_with_length_penalty(candidate, stimulus):
    c_norm = normalize(candidate)
    s_norm = normalize(stimulus)
    if not c_norm and not s_norm: return 0.0
    if not c_norm or not s_norm:  return 1.0
    base_dist = Levenshtein.distance(c_norm, s_norm) / max(len(c_norm), len(s_norm))
    len_ratio = len(c_norm) / max(len(s_norm), 1)
    if len_ratio > 1.5:
        length_penalty = 0.35 * min(len_ratio - 1.5, 1.5)
        return min(1.0, base_dist + length_penalty)
    return base_dist

edit_ratio = edit_ratio_with_length_penalty
print("✓ Functions defined")

✓ Functions defined


In [13]:
# =============================================================================
# CELL 8 — Quality analysis: distinguishing ASR errors from learner errors
# =============================================================================
# The task instruction: "correct only ASR errors, not participant grammar/
# vocabulary errors." This requires us to flag which deviations from the target
# are likely Whisper mishears vs. genuine learner production differences.
#
# Our approach uses two signals:
#
# 1. PHONETIC SIMILARITY
#    If a transcribed word is phonetically very close to the corresponding target
#    word but spelled differently, that's a strong signal of an ASR substitution
#    error (the learner said the right thing, Whisper heard the wrong thing).
#    We implement a lightweight Spanish phonetic normalization that collapses the
#    most common phoneme-grapheme ambiguities in Spanish:
#      b/v, ll/y, c/z/s, qu/k, h (silent), gu before e/i, rr/r, x, accent marks.
#
# 2. WHISPER CONFIDENCE (avg_logprob per token)
#    Low avg_logprob in a segment indicates Whisper itself wasn't sure —
#    these segments deserve closer human review regardless of edit distance.
#
# OUTPUT FLAGS:
#   HIGH   — transcript closely matches target; likely accurate
#   MEDIUM — moderate deviation; probably learner errors (preserve as-is)
#   LOW    — large deviation OR low Whisper confidence; human review recommended
#   CHECK  — possible ASR substitution: word is phonetically close to target
#             but orthographically different (most likely spot for ASR correction)
#   SILENT — segment was [inaudible]

_PHONETIC_RULES = [
    (r"qu",       "k"),
    (r"c([ei])",  r"s\1"),
    (r"c",        "k"),
    (r"z",        "s"),
    (r"v",        "b"),
    (r"ll",       "y"),
    (r"x",        "ks"),
    (r"h",        ""),
    (r"gu([ei])", r"g\1"),
    (r"rr",       "r"),
    (r"ñ",        "ny"),
]

def phonetic_key(word):
    """Simplified Spanish phonetic normalization for substitution detection."""
    w = normalize(word)
    for pattern, repl in _PHONETIC_RULES:
        w = re.sub(pattern, repl, w)
    return w


def find_asr_suspect_words(transcription_text, target_text):
    """
    Compare words between transcription and target.
    Return a list of (transcribed_word, target_word) pairs where the transcribed
    word is phonetically similar to the target word but not a match.
    This is where a human reviewer should check for ASR substitution errors.
    """
    trans_words  = [w for w in re.sub(r"[¿?¡!,\.]", "", transcription_text).split() if w]
    target_words = [w for w in re.sub(r"[¿?¡!,\.()\d]", "", target_text).split() if w]

    suspects = []
    # Simple 1-to-1 alignment based on position (good enough for short EIT sentences)
    for tw, rv in zip(trans_words, target_words):
        norm_tw = normalize(tw)
        norm_rv = normalize(rv)
        if norm_tw == norm_rv:
            continue  # exact match (accent-insensitive)

        phon_tw  = phonetic_key(tw)
        phon_rv  = phonetic_key(rv)
        phon_sim = 1.0 - Levenshtein.distance(phon_tw, phon_rv) / max(len(phon_tw), len(phon_rv), 1)

        # If phonetically very similar (>0.75) but not orthographically identical:
        # this is probably Whisper mishearing a correctly-produced word.
        if phon_sim > 0.75:
            suspects.append((tw, rv, round(phon_sim, 2)))

    return suspects


def assign_confidence(seg_data, target_text, edit_dist):
    """
    Assign a confidence label based on edit distance and Whisper internal metrics.
    """
    if seg_data["text"] == "[inaudible]":
        return "SILENT"

    avg_lp = seg_data.get("avg_logprob") or -0.5  # default if unavailable
    low_model_confidence = avg_lp < -0.9           # Whisper wasn't sure about this segment

    if edit_dist < 0.20:
        return "LOW" if low_model_confidence else "HIGH"
    elif edit_dist < 0.50:
        return "LOW" if low_model_confidence else "MEDIUM"
    else:
        return "LOW"   # major deviation always warrants human review


def build_participant_df(pid, aligned_segs):
    """Assemble the full results DataFrame for one participant."""
    rows = []
    for i, (seg, (stim_text, n_syllables)) in enumerate(zip(aligned_segs, TARGET_STIMULI)):
        transcript  = seg["text"]
        dist        = edit_ratio(transcript, stim_text) if transcript != "[inaudible]" else 1.0
        confidence  = assign_confidence(seg, stim_text, dist)
        suspects    = find_asr_suspect_words(transcript, stim_text) if confidence != "SILENT" else []

        # If we found phonetically-similar-but-different words, upgrade flag
        if suspects and confidence in ("MEDIUM", "HIGH"):
            confidence = "CHECK"   # worth a second look for ASR substitution

        rows.append({
            "Sentence":              i + 1,
            "Stimulus":              stim_text,
            "Syllables":             n_syllables,
            "Transcription":         transcript,
            "Confidence":            confidence,
            "Edit_Dist_to_Target":   round(dist, 4),
            "ASR_Suspect_Words":     "; ".join(f"{t}→{r}({s})"
                                               for t, r, s in suspects) if suspects else "",
            "Whisper_AvgLogProb":    seg.get("avg_logprob"),
            "Whisper_NoSpeechProb":  seg.get("no_speech_prob"),
            "DP_Cost":               seg.get("dp_cost"),
            "Start_Sec":             seg.get("start"),
            "End_Sec":               seg.get("end"),
            "N_Segs_Merged":         seg.get("n_segs_merged", 0),
        })

    return pd.DataFrame(rows)


print("Running quality analysis...\n" + "─" * 50)
results_dfs = {}
for pid, aligned in all_aligned.items():
    df = build_participant_df(pid, aligned)
    results_dfs[pid] = df

    print(f"\nParticipant {pid}:")
    for label in ["HIGH", "CHECK", "MEDIUM", "LOW", "SILENT"]:
        n = (df["Confidence"] == label).sum()
        bar = "█" * n
        print(f"  {label:<7} {n:>2}/30  {bar}")

print("\n" + "─" * 50)
print("Quality analysis done.")

Running quality analysis...
──────────────────────────────────────────────────

Participant 38010:
  HIGH    12/30  ████████████
  CHECK    0/30  
  MEDIUM  11/30  ███████████
  LOW      1/30  █
  SILENT   6/30  ██████

Participant 38011:
  HIGH     7/30  ███████
  CHECK    0/30  
  MEDIUM   6/30  ██████
  LOW     16/30  ████████████████
  SILENT   1/30  █

Participant 38012:
  HIGH     3/30  ███
  CHECK    1/30  █
  MEDIUM   7/30  ███████
  LOW     16/30  ████████████████
  SILENT   3/30  ███

Participant 38015:
  HIGH    11/30  ███████████
  CHECK    4/30  ████
  MEDIUM  13/30  █████████████
  LOW      2/30  ██
  SILENT   0/30  

──────────────────────────────────────────────────
Quality analysis done.


In [14]:
# =============================================================================
# CELL 9 — Write transcriptions back into the original Excel template
# =============================================================================
# We fill column C (Transcription) for each participant sheet.
# We color-code the cells by confidence level so the human reviewer knows
# exactly which sentences need a closer look without reading the whole sheet.
#
# Color scheme (chosen for legibility in both screen and print):
#   HIGH   → light green  (probably fine)
#   CHECK  → light orange (possible ASR substitution — verify the flagged word)
#   MEDIUM → light yellow (learner error probably, but review the transcription)
#   LOW    → light red    (low confidence — must be reviewed before use)
#   SILENT → light gray   (no audible response detected)

COLORS = {
    "HIGH":   "C6EFCE",   # green
    "CHECK":  "FFCC99",   # orange
    "MEDIUM": "FFEB9C",   # yellow
    "LOW":    "FFC7CE",   # red/pink
    "SILENT": "D9D9D9",   # gray
}

# Copy the original template to preserve its formatting (fonts, borders, etc.)
shutil.copy("/content/AutoEIT_Sample_Audio_Transcriptions.xlsx", OUTPUT_EXCEL)
wb = load_workbook(OUTPUT_EXCEL)

for pid, df in results_dfs.items():
    sheet_name = PARTICIPANTS[pid]["sheet"]
    if sheet_name not in wb.sheetnames:
        print(f"  WARNING: sheet '{sheet_name}' not found in workbook — skipping")
        continue

    ws = wb[sheet_name]
    print(f"  Writing {sheet_name} ...", end=" ")

    for _, row in df.iterrows():
        excel_row  = int(row["Sentence"]) + 1    # row 1 is header, data starts row 2
        confidence = row["Confidence"]
        transcript = row["Transcription"]

        # ── Column C: Transcription ──
        cell_c = ws.cell(row=excel_row, column=3)
        cell_c.value     = transcript
        cell_c.fill      = PatternFill(start_color=COLORS[confidence],
                                       end_color=COLORS[confidence],
                                       fill_type="solid")
        cell_c.alignment = Alignment(wrap_text=True)

        # ── Column D: Reviewer notes (confidence + flagged words) ──
        # This column was empty in the template — we use it for reviewer metadata.
        # A real scorer can delete column D before analysis if preferred.
        cell_d = ws.cell(row=excel_row, column=4)
        note   = f"[{confidence}]"
        if row["ASR_Suspect_Words"]:
            note += f" | CHECK: {row['ASR_Suspect_Words']}"
        cell_d.value = note
        cell_d.font  = Font(italic=True, color="595959", size=8)

    print("done")

wb.save(OUTPUT_EXCEL)
print(f"\n✓ Filled workbook saved as '{OUTPUT_EXCEL}'")

# Auto-download
files.download(OUTPUT_EXCEL)

  Writing 38010-2A ... done
  Writing 38011-1A ... done
  Writing 38012-2A ... done
  Writing 38015-1A ... done

✓ Filled workbook saved as 'AutoEIT_Transcriptions_FILLED.xlsx'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# =============================================================================
# CELL 10 — Full transcription report (CSV) + console summary
# =============================================================================

print("=" * 70)
print("  AUTOEIT TRANSCRIPTION QUALITY REPORT")
print("=" * 70)

for pid, df in results_dfs.items():
    version = PARTICIPANTS[pid]["version"]
    print(f"\n{'─' * 70}")
    print(f"  PARTICIPANT {pid}  (EIT version {version})")
    print(f"{'─' * 70}")

    total_transcribed = (df["Confidence"] != "SILENT").sum()
    avg_edit          = df.loc[df["Confidence"] != "SILENT", "Edit_Dist_to_Target"].mean()

    print(f"  Sentences transcribed:  {total_transcribed}/30")
    print(f"  Avg edit dist (≠SILENT):{avg_edit:.3f}")
    print(f"  (0 = identical to stimulus; high values = learner deviated, which is normal)")

    print(f"\n  {'#':>3}  {'Stimulus':<44}  {'Transcription':<44}  Conf")
    print(f"  {'─' * 103}")
    for _, row in df.iterrows():
        s = row["Stimulus"][:42]
        t = row["Transcription"][:42]
        c = row["Confidence"]
        print(f"  {int(row['Sentence']):>3}  {s:<44}  {t:<44}  {c}")

# Combine all participants into a single CSV for researchers
all_frames = []
for pid, df in results_dfs.items():
    tmp = df.copy()
    tmp.insert(0, "Participant_ID", pid)
    tmp.insert(1, "EIT_Version", PARTICIPANTS[pid]["version"])
    all_frames.append(tmp)

full_report = pd.concat(all_frames, ignore_index=True)

# utf-8-sig adds BOM so Excel opens the CSV without encoding issues on Windows
full_report.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"\n✓ Full report saved as '{OUTPUT_CSV}'")
files.download(OUTPUT_CSV)

print("\n✓ All outputs downloaded. Pipeline complete.")

  AUTOEIT TRANSCRIPTION QUALITY REPORT

──────────────────────────────────────────────────────────────────────
  PARTICIPANT 38010  (EIT version 2A)
──────────────────────────────────────────────────────────────────────
  Sentences transcribed:  24/30
  Avg edit dist (≠SILENT):0.226
  (0 = identical to stimulus; high values = learner deviated, which is normal)

    #  Stimulus                                      Transcription                                 Conf
  ───────────────────────────────────────────────────────────────────────────────────────────────────────
    1  Quiero cortarme el pelo                       Quiero cortarme el pelo.                      HIGH
    2  El libro está en la mesa                      El libro está en la mesa.                     HIGH
    3  El carro lo tiene Pedro                       [inaudible]                                   SILENT
    4  El se ducha cada mañana                       El carro no tiene pelo. Él se duche cada m    LOW
    5  ¿Q

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ All outputs downloaded. Pipeline complete.


In [16]:
# =============================================================================
# CELL 11 — Optional: WER / CER evaluation against target sentences
# =============================================================================
# NOTE ON INTERPRETATION:
#
# For typical ASR evaluation, high WER means the model failed.
# For EIT, high WER can mean the LEARNER failed to reproduce the stimulus —
# which is exactly the signal the researcher wants to capture.
# So we report these metrics not to evaluate our pipeline's accuracy,
# but as a proxy for learner proficiency (lower WER → closer to target).
#
# A separate validation would require the gold-standard human transcription
# (which is what this task is asking us to produce, so we don't have it).
# If a gold transcription is available, replace `reference` with it below.

from jiwer import wer, cer, transforms

# Normalize transform: lowercase, strip punctuation, collapse spaces
jiwer_transform = transforms.Compose([
    transforms.ToLowerCase(),
    transforms.RemovePunctuation(),
    transforms.Strip(),
    transforms.ReduceToListOfListOfWords(),
])

print("=" * 60)
print("  WER / CER vs. Target Stimuli")
print("  (measures learner deviation, NOT pipeline accuracy)")
print("=" * 60)

for pid, df in results_dfs.items():
    # Exclude [inaudible] responses from WER calculation
    valid = df[df["Confidence"] != "SILENT"]
    if valid.empty:
        print(f"\n{pid}: no valid segments, skipping.")
        continue

    references   = valid["Stimulus"].tolist()
    hypotheses   = valid["Transcription"].tolist()

    # Accent-insensitive comparison: strip combining diacritics
    def accent_strip(text):
        nfkd = unicodedata.normalize("NFKD", text)
        return "".join(c for c in nfkd if not unicodedata.combining(c))

    refs_clean  = [accent_strip(r) for r in references]
    hyps_clean  = [accent_strip(h) for h in hypotheses]

    word_error_rate = wer(refs_clean, hyps_clean)
    char_error_rate = cer(refs_clean, hyps_clean)

    print(f"\nParticipant {pid} ({len(valid)}/30 non-silent):")
    print(f"  WER (vs. target): {word_error_rate:.2%}")
    print(f"  CER (vs. target): {char_error_rate:.2%}")

print("""
─────────────────────────────────────────────────────────────
Reminder: WER here = deviation from stimulus, not ASR error.
L2 learners at lower proficiency will naturally show higher WER.
This metric should be compared across participants, not against
an 'acceptable' absolute threshold.
─────────────────────────────────────────────────────────────
""")

  WER / CER vs. Target Stimuli
  (measures learner deviation, NOT pipeline accuracy)

Participant 38010 (24/30 non-silent):
  WER (vs. target): 43.22%
  CER (vs. target): 26.60%

Participant 38011 (29/30 non-silent):
  WER (vs. target): 75.10%
  CER (vs. target): 55.56%

Participant 38012 (27/30 non-silent):
  WER (vs. target): 79.82%
  CER (vs. target): 56.84%

Participant 38015 (30/30 non-silent):
  WER (vs. target): 42.57%
  CER (vs. target): 24.82%

─────────────────────────────────────────────────────────────
Reminder: WER here = deviation from stimulus, not ASR error.
L2 learners at lower proficiency will naturally show higher WER.
This metric should be compared across participants, not against
an 'acceptable' absolute threshold.
─────────────────────────────────────────────────────────────



---
## Approach Description & Challenges

### Pipeline Architecture

| Stage | Tool / Method | Why |
|-------|--------------|-----|
| Audio prep | pydub + librosa + noisereduce | Offset trim, 16kHz mono, mild denoising |
| ASR | Whisper large-v3 (faster-whisper) | Best L2 Spanish accuracy, zero API cost |
| Alignment | DP sequence alignment | Preserves EIT order; handles over/under-segmentation |
| Quality flags | Phonetic similarity + logprob | Separates ASR errors from learner errors |
| Output | openpyxl | Fills column C of provided template with color-coded confidence |

### Key Design Decisions

**`condition_on_previous_text=False`** — The single most important setting for EIT transcription. With this on (Whisper's default), the language model prefix from sentence *n* biases decoding of sentence *n+1* toward what fluently follows in Spanish. This systematically "corrects" the learner's actual output toward a plausible continuation of the target text — precisely the behavior we need to suppress.

**DP alignment over fuzzy matching** — The original approach (thefuzz `extractOne`) matches each Whisper segment to whichever target is most similar, ignoring order. For EIT this fails badly: a lower-proficiency learner might produce output that's equally (un)similar to multiple stimuli, and the fuzzy matcher will pick the wrong one. The DP alignment encodes the sequential constraint as a hard invariant.

**`temperature=0.0`** — Whisper's temperature fallback mechanism (retrying with increasing temperature when decoding gets stuck) was designed for fluent speech. The long silences between EIT responses are exactly the context where higher temperatures cause hallucination. We keep temperature fixed at 0 and let the compression-ratio filter reject any hallucinated segments.

**No paid API** — 100% local inference via faster-whisper. Model weights download once (~3 GB) and run entirely on Colab's free T4 GPU. No Gemini, no OpenAI, no per-token cost.

### Challenges Encountered

1. **Participant 038012 — 12-minute preamble.** The Excel note warns that the first 12 minutes contain non-EIT audio. We trim this before any processing. Without this trim, Whisper would burn ~50% of its context window on non-speech data and potentially run out of memory for the actual responses.

2. **Over-segmentation of disfluent responses.** L2 learners frequently pause mid-sentence ("El libro... está... en la mesa"). Whisper's VAD treats these as sentence boundaries and splits what is one response into 2-3 segments. Our merge step with a 3.2-second gap threshold addresses this — the threshold was chosen to be longer than a normal inter-word pause but shorter than a true inter-stimulus interval.

3. **Distinguishing ASR errors from learner errors.** This is the fundamental unsolved problem. Our phonetic similarity flag (`CHECK` confidence) is a heuristic, not a classifier. Cases where both the ASR and the learner could plausibly have made the same substitution (e.g., a vowel that sounds similar in both the learner's L1 and Spanish) are genuinely ambiguous without listening to the audio. The `CHECK` flag is a signal to the human reviewer, not a decision.

4. **Evaluating output quality without a gold standard.** We report edit distance to the *target stimulus* rather than to a human transcription, because we don't have the latter (that's what we're producing). This means we can't compute true Word Error Rate for our ASR pipeline. The metrics in Cell 11 reflect learner deviation from the target, which is the actual quantity of interest for EIT scoring, not ASR accuracy.

5. **Spanish accent marks in Whisper output.** Whisper large-v3 generally handles Spanish orthography well but occasionally drops or misplaces accent marks (e.g., `esta` vs `está`). Our normalization layer strips combining diacritics before edit-distance calculations so that `esta/está` are treated as identical — because a human transcriber might also omit an accent mark without it affecting the linguistic interpretation.

### How to Evaluate the Output

1. Open `AutoEIT_Transcriptions_FILLED.xlsx`. Cells are color-coded: **green = HIGH confidence, orange = CHECK (possible ASR substitution), yellow = MEDIUM, red = LOW, gray = no speech detected.**
2. Start with **LOW** cells — these are most likely to need correction.
3. For **CHECK** cells, column D shows the specific word pair flagged (e.g., `cortarme→cortar(0.91)` means the transcribed word `cortarme` is 91% phonetically similar to the target `cortar` but differs — was this an ASR mishear or a learner omission?).
4. For **SILENT** cells, the participant may have produced an inaudible response — worth listening to the audio at the indicated timestamp.
5. Do **not** correct MEDIUM or HIGH cells based solely on similarity to the target — if the learner said something different, the transcription should reflect that.